## Challenge two: BigQuery ML: Predicting Emergency Call Response Times

### Importing big query from Google Cloud library

In [18]:
from google.cloud import bigquery

### variables

In [19]:
import os

# --- Diagnostic: shows what each detection method returns ---
print("=== Project detection diagnostic ===")
print("env GOOGLE_CLOUD_PROJECT:", os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("env GCP_PROJECT         :", os.environ.get("GCP_PROJECT"))
try:
    import google.auth
    _creds, _proj = google.auth.default()
    print("google.auth project     :", _proj)
except Exception as e:
    print("google.auth failed      :", e)
print("=" * 36)


def detect_project_id():
    # 1. Explicit env var wins (lets anyone override without editing code)
    env = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT")
    if env:
        return env
    # 2. Ask Application Default Credentials what project we're running under
    try:
        import google.auth
        _, project = google.auth.default()
        if project:
            return project
    except Exception:
        pass
    # 3. Last resort: query the metadata server (works on GCP runtimes)
    try:
        import urllib.request
        req = urllib.request.Request(
            "http://metadata.google.internal/computeMetadata/v1/project/project-id",
            headers={"Metadata-Flavor": "Google"},
        )
        return urllib.request.urlopen(req, timeout=2).read().decode()
    except Exception:
        return None


class Config:
    def __init__(
        self,
        project_id=None,                      # None -> auto-detect from the runtime
        dataset="emergency",
        location="US",                        # must match the GCS bucket region
        source_uri="gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv",
        raw_table_name="emergency_calls_raw",
        model_name="response_time_model",
        label_column="response_time",
    ):
        resolved = project_id or detect_project_id()
        if not resolved:
            raise ValueError(
                "Could not determine project_id. Pass it explicitly: "
                "Config(project_id='my-project')"
            )
        # bypass our own __setattr__ block during construction
        object.__setattr__(self, "project_id", resolved)
        object.__setattr__(self, "dataset", dataset)
        object.__setattr__(self, "location", location)
        object.__setattr__(self, "source_uri", source_uri)
        object.__setattr__(self, "raw_table_name", raw_table_name)
        object.__setattr__(self, "model_name", model_name)
        object.__setattr__(self, "label_column", label_column)

    def __setattr__(self, name, value):
        raise AttributeError(f"Config is immutable; can't set {name!r}")

    @property
    def raw_table(self):
        return f"{self.project_id}.{self.dataset}.{self.raw_table_name}"

    @property
    def model(self):
        return f"{self.project_id}.{self.dataset}.{self.model_name}"


# --- Build the config ---
# Auto-detects on a GCP runtime. If detection fails, either:
#   - set os.environ["GOOGLE_CLOUD_PROJECT"] = "your-project" above, OR
#   - pass it directly: Config(project_id="your-project")
CFG = Config()

print("\nProject  :", CFG.project_id)
print("Raw table:", CFG.raw_table)
print("Model    :", CFG.model)

=== Project detection diagnostic ===
env GOOGLE_CLOUD_PROJECT: qwiklabs-gcp-01-5fe45b5e4e14
env GCP_PROJECT         : None
google.auth project     : qwiklabs-gcp-01-5fe45b5e4e14

Project  : qwiklabs-gcp-01-5fe45b5e4e14
Raw table: qwiklabs-gcp-01-5fe45b5e4e14.emergency.emergency_calls_raw
Model    : qwiklabs-gcp-01-5fe45b5e4e14.emergency.response_time_model


### Logging

In [20]:
import logging, json, sys, uuid, datetime as dt

logger = logging.getLogger("response_time_ml")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

RUN_ID = str(uuid.uuid4())

def log_event(step: str, status: str, **kw):
    logger.info(json.dumps({"run_id": RUN_ID, "step": step, "status": status, **kw}))

log_event("init", "ok", started=dt.datetime.now(dt.timezone.utc).isoformat())

2026-06-02 17:15:48,314 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "init", "status": "ok", "started": "2026-06-02T17:15:48.314176+00:00"}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "init", "status": "ok", "started": "2026-06-02T17:15:48.314176+00:00"}


### Model Training

In [21]:
class ResponseTimeModel:
    """Loads raw emergency-call data, trains a BigQuery ML regression model, and uses it."""

    MODEL_SQL = """
    CREATE OR REPLACE MODEL `{model}`
    OPTIONS (
      model_type = 'linear_reg',
      input_label_cols = ['{label}'],
      data_split_method = 'AUTO_SPLIT',
      enable_global_explain = TRUE
    ) AS
    SELECT
      call_type,
      location,
      weather_condition,
      day_of_week,
      time_of_day,
      traffic_level,
      distance_to_station,
      units_available,
      {label}
    FROM `{raw_table}`
    """

    PREDICT_SQL = """
    SELECT *
    FROM ML.PREDICT(MODEL `{model}`, (
      SELECT
        'Rescue' AS call_type, 'Maplewood' AS location, 'Sunny' AS weather_condition,
        'Tuesday' AS day_of_week, 9 AS time_of_day, 'Low' AS traffic_level,
        2.5 AS distance_to_station, 6 AS units_available
      UNION ALL
      SELECT
        'Police', 'Downtown', 'Snowy',
        'Friday', 23, 'High',
        18.0, 1
    ))
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.client = bigquery.Client(project=cfg.project_id, location=cfg.location)
        self.rows_in = 0

    def ensure_dataset(self):
        ds = bigquery.Dataset(f"{self.cfg.project_id}.{self.cfg.dataset}")
        ds.location = self.cfg.location
        self.client.create_dataset(ds, exists_ok=True)
        log_event("dataset_ready", "ok", dataset=self.cfg.dataset)

    def load_raw(self):
        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        )
        job = self.client.load_table_from_uri(
            self.cfg.source_uri, self.cfg.raw_table, job_config=job_config
        )
        job.result()
        self.rows_in = self.client.get_table(self.cfg.raw_table).num_rows
        log_event("load_raw", "ok", rows=self.rows_in, job_id=job.job_id)

    def train(self):
        sql = self.MODEL_SQL.format(
            model=self.cfg.model, raw_table=self.cfg.raw_table, label=self.cfg.label_column
        )
        job = self.client.query(sql)
        job.result()
        log_event("train", "ok", model=self.cfg.model_name, job_id=job.job_id)

    def preview_raw(self, n: int = 10):
        df = self.client.query(
            f"SELECT * FROM `{self.cfg.raw_table}` LIMIT {n}"
        ).to_dataframe()
        log_event("preview_raw", "ok", rows=len(df))
        return df

    def profile(self):
        df = self.client.query(f"""
        SELECT
          COUNT(*) AS row_count,
          ROUND(AVG({self.cfg.label_column}), 2) AS avg_response,
          ROUND(MIN({self.cfg.label_column}), 2) AS min_response,
          ROUND(MAX({self.cfg.label_column}), 2) AS max_response,
          ROUND(STDDEV({self.cfg.label_column}), 2) AS stddev_response
        FROM `{self.cfg.raw_table}`
        """).to_dataframe()
        log_event("profile", "ok")
        return df

    def label_by_call_type(self):
        df = self.client.query(f"""
        SELECT
          call_type,
          COUNT(*) AS calls,
          ROUND(AVG({self.cfg.label_column}), 2) AS avg_response
        FROM `{self.cfg.raw_table}`
        GROUP BY call_type
        ORDER BY avg_response DESC
        """).to_dataframe()
        log_event("label_by_call_type", "ok", groups=len(df))
        return df

    def training_features(self, n: int = 10):
        """The exact columns fed into the model — what training actually sees."""
        df = self.client.query(f"""
        SELECT
          call_type, location, weather_condition, day_of_week,
          time_of_day, traffic_level, distance_to_station,
          units_available, {self.cfg.label_column}
        FROM `{self.cfg.raw_table}`
        LIMIT {n}
        """).to_dataframe()
        log_event("training_features", "ok", rows=len(df))
        return df

    def evaluate(self):
        job = self.client.query(f"SELECT * FROM ML.EVALUATE(MODEL `{self.cfg.model}`)")
        df = job.result().to_dataframe()
        log_event("evaluate", "ok", job_id=job.job_id)
        return df

    def predict(self):
        sql = self.PREDICT_SQL.format(model=self.cfg.model)
        job = self.client.query(sql)
        df = job.result().to_dataframe()
        log_event("predict", "ok", rows=len(df), job_id=job.job_id)
        return df

    def run(self):
        self.ensure_dataset()
        self.load_raw()
        self.train()
        log_event("run", "ok", rows_in=self.rows_in)

### Setup: dataset + load raw data

In [22]:
pipeline = ResponseTimeModel(CFG)
pipeline.ensure_dataset()
pipeline.load_raw()
print(f"Loaded {pipeline.rows_in:,} rows into {CFG.raw_table}")

2026-06-02 17:15:48,745 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "dataset_ready", "status": "ok", "dataset": "emergency"}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "dataset_ready", "status": "ok", "dataset": "emergency"}


2026-06-02 17:15:52,358 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "load_raw", "status": "ok", "rows": 50000, "job_id": "f8a13cb2-4cf2-4567-a42f-d452e8a10b67"}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "load_raw", "status": "ok", "rows": 50000, "job_id": "f8a13cb2-4cf2-4567-a42f-d452e8a10b67"}


Loaded 50,000 rows into qwiklabs-gcp-01-5fe45b5e4e14.emergency.emergency_calls_raw


### Preview the raw data

In [23]:
pipeline.preview_raw()

2026-06-02 17:15:54,315 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "preview_raw", "status": "ok", "rows": 10}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "preview_raw", "status": "ok", "rows": 10}


,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37
5,17582,2023-01-01 02:28:50+00:00,Rescue,Downtown,Snowy,Sunday,2,High,25.15,6,28.48
6,21624,2023-01-01 02:44:06+00:00,Rescue,Oakmont,Snowy,Sunday,2,High,3.95,9,19.30
7,36793,2023-01-01 02:53:54+00:00,Fire,Riverside,Sunny,Sunday,2,High,5.87,10,10.72
8,41350,2023-01-01 03:52:33+00:00,Police,Greenfield,Windy,Sunday,3,High,6.66,5,20.55
9,32092,2023-01-01 04:09:23+00:00,Police,Maplewood,Snowy,Sunday,4,High,15.50,13,22.98


### Profile the response_time we are predicting

In [24]:
pipeline.profile()

2026-06-02 17:15:56,510 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "profile", "status": "ok"}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "profile", "status": "ok"}


,row_count,avg_response,min_response,max_response,stddev_response
0,50000,17.45,2.01,36.55,5.3


### Average response time by call type

In [25]:
pipeline.label_by_call_type()

2026-06-02 17:15:58,471 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "label_by_call_type", "status": "ok", "groups": 4}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "label_by_call_type", "status": "ok", "groups": 4}


,call_type,calls,avg_response
0,Police,12536,17.51
1,Rescue,12401,17.44
2,Fire,12585,17.43
3,Medical,12478,17.39


### The exact features going into the model

In [26]:
pipeline.training_features()

2026-06-02 17:16:01,301 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "training_features", "status": "ok", "rows": 10}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "training_features", "status": "ok", "rows": 10}


,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37
5,Rescue,Downtown,Snowy,Sunday,2,High,25.15,6,28.48
6,Rescue,Oakmont,Snowy,Sunday,2,High,3.95,9,19.30
7,Fire,Riverside,Sunny,Sunday,2,High,5.87,10,10.72
8,Police,Greenfield,Windy,Sunday,3,High,6.66,5,20.55
9,Police,Maplewood,Snowy,Sunday,4,High,15.50,13,22.98


### Train the model — CREATE MODEL

In [27]:
pipeline.train()
print(f"Model {CFG.model} trained.")

2026-06-02 17:16:22,907 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "train", "status": "ok", "model": "response_time_model", "job_id": "bad852ff-ace0-4dea-ae68-0b5496189f69"}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "train", "status": "ok", "model": "response_time_model", "job_id": "bad852ff-ace0-4dea-ae68-0b5496189f69"}


Model qwiklabs-gcp-01-5fe45b5e4e14.emergency.response_time_model trained.


### Evaluation — ML.EVALUATE

In [28]:
pipeline.evaluate()

2026-06-02 17:16:25,050 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "evaluate", "status": "ok", "job_id": "b2197928-f20a-48b0-ade7-0dfd2f5dd3c8"}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "evaluate", "status": "ok", "job_id": "b2197928-f20a-48b0-ade7-0dfd2f5dd3c8"}


,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.761934,4.827846,0.015117,1.501836,0.831417,0.83146


### Prediction on synthetic data — ML.PREDICT

In [29]:
pipeline.predict()

2026-06-02 17:16:27,195 | INFO | {"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "predict", "status": "ok", "rows": 2, "job_id": "3cce7ac6-1a6f-4004-8051-759bcffbba73"}


INFO:response_time_ml:{"run_id": "533d9582-25b0-497a-889d-5ecc0e5ad469", "step": "predict", "status": "ok", "rows": 2, "job_id": "3cce7ac6-1a6f-4004-8051-759bcffbba73"}


,predicted_response_time,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available
0,7.252596,Rescue,Maplewood,Sunny,Tuesday,9,Low,2.5,6
1,26.157526,Police,Downtown,Snowy,Friday,23,High,18.0,1
